# DALL-E — Zero-Shot Text-to-Image Generation (2021)_Papers · Generative Models_**Compress an image into discrete tokens, then let a transformer write text-then-image as one stream.**---This notebook builds DALL-E's two mechanics one step at a time: the dVAE compression factor, a toy codebook that turns continuous patches into discrete tokens, and a toy autoregressive sampler that draws tokens one at a time. Run each cell, read the note above it, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.import numpy as npimport matplotlib.pyplot as plt

## Reference implementation — NumPyDALL-E has two moving parts we can illustrate on a toy scale: (1) a dVAE that **compresses** an image into a small grid of discrete tokens, and (2) an **autoregressive** transformer that writes text-then-image tokens one at a time. We build it in three steps: the compression factor the paper quotes, a tiny codebook quantizer, and a tiny next-token sampler.

### Step 1 — Compute the dVAE compression factorThe paper compresses a 256x256 image into a 32x32 grid of tokens. The number of *positions* shrinks by 64x; folding in the 3 RGB channels (a pixel is 3 numbers, a grid cell is 1 token) gives the 192x reduction the paper reports (Section 2, Stage One). These are quoted sizes, not a training run.

In [ ]:
import numpy as np# The paper's stated compression factor: 256x256 image -> 32x32 token grid.# Positions shrink 64x; folding in 3 color channels gives the 192x the paper reports.pixels = 256 * 256           # 65536 pixel positionstokens = 32 * 32             # 1024 token-grid positionspos_factor = pixels // tokens        # 64full_factor = pos_factor * 3         # 64 * 3 = 192 (three RGB channels)print("pixel positions      =", pixels)print("token positions      =", tokens)print("position factor      =", pos_factor)print("with 3 channels      =", full_factor, "(paper states 192)")# pixel positions      = 65536# token positions      = 1024# position factor      = 64# with 3 channels      = 192 (paper states 192)

### Step 2 — Quantize continuous patches into discrete tokensThe dVAE's job is to snap each continuous image patch to the nearest entry of a learned **codebook**, turning it into a discrete token index. Here the codebook is a tiny fixed 5-entry set so you can see the *form* of the operation (nearest-neighbour snapping), not the paper's learned 8192-entry codebook.

In [ ]:
# OUR toy CODEBOOK QUANTIZATION: continuous "patch" values -> discrete tokens.# A real dVAE LEARNS its codebook; here it is a tiny fixed one. Illustration of the FORM only.codebook = np.array([0.0, 0.25, 0.5, 0.75, 1.0])      # 5 made-up entriespatches = np.array([0.05, 0.30, 0.52, 0.71, 0.98])    # 5 continuous patch values# Snap each patch to the nearest codebook entry -> its discrete token index.distances = np.abs(patches[:, None] - codebook[None, :])tokens_idx = np.argmin(distances, axis=1)print("toy codebook quantization (illustration):")print("  patches      =", patches)print("  -> token idx =", tokens_idx)               # the discrete tokensprint("  -> values    =", codebook[tokens_idx])# toy codebook quantization (illustration):#   patches      = [0.05 0.3  0.52 0.71 0.98]#   -> token idx = [0 1 2 3 4]#   -> values    = [0.   0.25 0.5  0.75 1.  ]

### Step 3 — Sample image tokens autoregressivelyDALL-E's transformer draws image tokens **one at a time**, each conditioned on the tokens already drawn (and the text). We mimic that with a tiny fixed next-token table `P[prev]`: start from a token chosen by a "text" condition, then repeatedly sample the next token from the row for the previous one. A real transformer *learns* this distribution; here it is a made-up 4x4 table.

In [ ]:
# OUR toy AUTOREGRESSIVE SAMPLER over a 4-symbol image vocabulary, drawing tokens one at a# time conditioned on the previous token. Illustration of the FORM only, NOT the paper's transformer.rng = np.random.default_rng(0)vocab = ["A", "B", "C", "D"]# Fixed next-token probability table P[prev] -> distribution over the next token.# (A real transformer LEARNS this; here it is a made-up 4x4 table.)P = np.array([[0.1, 0.6, 0.2, 0.1],              [0.2, 0.1, 0.6, 0.1],              [0.1, 0.2, 0.1, 0.6],              [0.6, 0.1, 0.2, 0.1]])prev = 0                       # start token, chosen by a "text" conditionseq = [vocab[prev]]for _ in range(6):             # sample 6 image tokens one at a time    prev = rng.choice(4, p=P[prev])    seq.append(vocab[prev])print("toy autoregressive sample (each token depends on the previous):")print(" ", " -> ".join(seq))# toy autoregressive sample (each token depends on the previous):#   A -> B -> B -> A -> A -> C -> D# Continuous patches became discrete tokens; tokens were drawn left-to-right.# These are OUR toy numbers, NOT the paper's codebook, transformer, or results.

## Visualize the data & results_DALL-E's two moving parts are (a) a codebook that turns continuous image patches into DISCRETE tokens, and (b) an AUTOREGRESSIVE sampler that draws image tokens one at a time. On a tiny toy, can we see continuous values snap to a small set of token levels?_We replay the same two mechanics here, in two steps: the codebook snapping (printed per patch) and the autoregressive draw. Round, made-up constants — this shows the FORM, not the paper's scale.

### Step 1 — Show each patch snapping to its token levelSame quantizer as the reference step, but printed patch-by-patch so you can read off which discrete level each continuous value lands on. Every continuous patch collapses to one of the five codebook levels — the discreteness the transformer relies on.

In [ ]:
import numpy as np# (a) CODEBOOK QUANTIZATION: continuous patches -> nearest discrete token level.codebook = np.array([0.0, 0.25, 0.5, 0.75, 1.0])      # tiny 5-level codebookpatches = np.array([0.05, 0.30, 0.52, 0.71, 0.98])    # continuous toy patchesdistances = np.abs(patches[:, None] - codebook[None, :])tok = np.argmin(distances, axis=1)print("patch value -> token level (the discrete token):")for i, (p, t) in enumerate(zip(patches, tok)):    print("  patch[%d]=%.2f -> token %d  (level %.2f)" % (i, p, t, codebook[t]))# patch value -> token level (the discrete token):#   patch[0]=0.05 -> token 0  (level 0.00)#   patch[1]=0.30 -> token 1  (level 0.25)#   patch[2]=0.52 -> token 2  (level 0.50)#   patch[3]=0.71 -> token 3  (level 0.75)#   patch[4]=0.98 -> token 4  (level 1.00)

### Step 2 — Draw a short autoregressive token sequenceSame toy sampler as the reference step: start from token `A` and draw six more tokens, each conditioned on the previous one via the fixed next-token table. The left-to-right dependence is exactly what makes the generated image coherent rather than a bag of independent tokens.

In [ ]:
# (b) AUTOREGRESSIVE SAMPLING over a 4-symbol image vocabulary, one token at a time#     conditioned on the previous token (a made-up next-token table).rng = np.random.default_rng(0)vocab = ["A", "B", "C", "D"]P = np.array([[0.1, 0.6, 0.2, 0.1],              [0.2, 0.1, 0.6, 0.1],              [0.1, 0.2, 0.1, 0.6],              [0.6, 0.1, 0.2, 0.1]])prev = 0seq = ["A"]for _ in range(6):    prev = rng.choice(4, p=P[prev])    seq.append(vocab[prev])print("autoregressive sample (each token depends on the previous):")print(" ", " -> ".join(seq))# autoregressive sample (each token depends on the previous):#   A -> B -> B -> A -> A -> C -> D# Continuous patches became discrete tokens; tokens were sampled left-to-right.# OUR toy numbers -- NOT the paper's codebook, transformer, or reported results.

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** Compression factor. The dVAE turns a 256-by-256 RGB image into a
            32-by-32 grid of single tokens. (a) By what factor does the number of
            grid positions shrink versus one-token-per-pixel? (b) Folding in the
            three color channels (a pixel is three numbers, a cell is one token),
            what total reduction does the paper state?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Count positions: $256 \times 256 = 65536$ pixels vs. $32 \times 32 = 1024$ grid cells. — _The transformer's cost grows with sequence length, so the number of positions is what matters._
- Divide: $65536 / 1024 = 64$. — _That is the position-count reduction ignoring color._
- A pixel carries 3 channel numbers but a cell carries 1 token, an extra $\times 3$: $64 \times 3 = 192$. — _The paper states the dVAE 'reduces the context size of the transformer by a factor of 192' (&sect;2, Stage One)._

**Answer:** (a) $65536 / 1024 = 64$ &mdash; the token grid has 64 times fewer
                 positions than one-token-per-pixel. (b) Including the three color
                 channels, $64 \times 3 = 192$, matching the paper's stated factor
                 of 192 (&sect;2). That is why a stream that would be ~196k raw
                 numbers becomes at most ~1280 token positions the transformer can
                 actually handle.

</details>

**Problem 2.** Generation order. You have a trained DALL-E and a new caption.
            Walk through how a picture is produced, in order. Where does the dVAE
            decoder come in, and why must the image tokens be sampled one at a
            time rather than all at once?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Feed the caption's text tokens to the transformer. — _The text conditions which image tokens are likely, so the picture matches the caption._
- Sample the 1024 image tokens autoregressively: token $t$ conditioned on the caption and tokens $1..t-1$. — _The transformer models a JOINT distribution; each token's distribution depends on the ones before it, which keeps the image coherent._
- Pass the completed 32-by-32 token grid to the dVAE decoder to paint a 256-by-256 image. — _The transformer only produces tokens; only the decoder maps tokens back to pixels._

**Answer:** Order: (1) feed the caption's text tokens; (2) sample the 1024
                 image tokens one at a time, each conditioned on the caption and the
                 tokens already drawn; (3) hand the finished grid to the dVAE
                 decoder, which renders the pixels. Sampling must be left-to-right
                 because the transformer is autoregressive &mdash; each token's
                 distribution depends on the previous tokens, so independent
                 sampling would lose that conditioning and produce an incoherent
                 grid. The decoder is the final, separate step that turns tokens
                 into a picture.

</details>

**Problem 3.** Ablation: kill the discrete bottleneck. Suppose you replace the
            dVAE's discrete tokens with the encoder's raw continuous activations
            and feed those directly to a transformer over text-then-image.
            Conceptually, what two things break, tying each to a specific part of
            the method?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Note the transformer (&sect;2.2) is a model over DISCRETE tokens from finite vocabularies (text 16384, image 8192). — _An autoregressive transformer predicts a categorical next-token distribution; it needs a finite alphabet to predict over._
- Continuous activations have no finite alphabet, so 'predict the next token' is undefined; you would need a different (continuous) output model. — _The whole point of the codebook is to give the image a discrete alphabet the language-model machinery can handle._
- Also, the Gumbel-softmax relaxation (&sect;2.1) exists precisely to train THROUGH the discrete choice; with no discrete choice there is nothing it is approximating, and the codebook-usage objective (the $\beta$-weighted KL) loses its meaning. — _Equation 1's agreement term pulls the discrete token distribution toward the prior; remove discreteness and that term changes entirely._

**Answer:** Two things break. (1) The autoregressive transformer predicts a
                 categorical distribution over a finite vocabulary; raw continuous
                 activations have no finite alphabet, so 'sample the next image
                 token' is undefined &mdash; you would need a different continuous
                 generator, losing the clean text-then-image token stream. (2) The
                 discrete bottleneck is exactly what the Gumbel-softmax relaxation
                 (&sect;2.1) and the codebook (8192 entries) are built around; the
                 ELB's $\beta$-weighted agreement term (Eq 1) is about matching a
                 distribution over discrete tokens, so removing discreteness
                 guts the objective. The discrete token grid is what makes a
                 language-model-style transformer applicable to images at all.

</details>